[Guide Index](README.md)


# 01 Getting Started

StarLayer is a Python library that adds capabilities from the current RDF/SPARQL/SHACL 1.2 draft specifications. Its graph class is a thin wrapper over an [Oxigraph](https://github.com/oxigraph/oxigraph) store (`pyoxigraph.Store`), and its SHACL validator is StarLayer's own engine. This guide walks through the four basic operations (install, graph construction, querying, validation); each section below links to the guide that covers its own topic in depth.

These guides assume a working knowledge of RDF 1.1, SPARQL 1.1, and SHACL 1.1. The graph class is `StarLayerGraph` (and `StarLayerDataset` for datasets). The [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb) covers the boundary between the wrapper and Oxigraph.

## How to run this notebook

Run cells from top to bottom - section 1 installs StarLayer if it isn't already installed, and each later section builds its own small, self-contained example.


## 1. Install

This is the supported public install path. It brings in the graph, SPARQL, and SHACL components together. If you're working from a local checkout for development instead, see the root [README](../../README.md)'s "Development" section.

Because StarLayer is under development and evolving in-line with the draft standards, it is helpful to periodically refresh the install from the github repository.  

In [1]:
try:
    import starlayer  
    print("starlayer is already installed - skipping install.")
except ImportError:
    %pip install "git+https://github.com/hidden-graph/starlayer.git"

### Getting a fresh version

# The check above skips installing if `starlayer` is already importable.
# It won't reflect subsequent changes. To force a clean reinstall from the repo, uncomment and run:

# %pip uninstall -y starlayer
# %pip install --no-cache-dir "git+https://github.com/hidden-graph/starlayer.git"


starlayer is already installed - skipping install.


In [2]:
from starlayer.graph.oxgraph import StarLayerGraph, StarLayerDataset
from starlayer.shacl import validate

EX = "http://example.org/"

## 2. Using `StarLayerGraph`

`StarLayerGraph` holds a `pyoxigraph.Store` (in memory unless you pass `path=`), and parsing, serializing and querying are carried out by Oxigraph. It is not an `rdflib.Graph`: the methods you know (`add`, `remove`, `triples`, `parse`, `serialize`, `query`, `update`, `bind`) exist with the same shape, but terms that come back out are pyoxigraph terms (`NamedNode`, `BlankNode`, `Literal`, `Triple`), not rdflib ones. rdflib terms are still accepted as input.

A new graph has no prefixes bound. Bind the ones you need with `bind()` or pass `prefixes=` to the constructor; they are handed to Oxigraph on each `serialize()` and `query()` call.

The [Graphs guide](02-graphs.ipynb) covers RDF 1.2-specific content (triple terms, reification, direction-tagged literals.).

The examples in the user guide are presented in the Turtle serialization format. The [serialization formats guide](06a-serialization-formats.ipynb) covers additional supported serialization formats.

In [3]:
# Construct the graph, and bind a prefix (a new graph has none).
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

print("triples:", len(g))
# Oxigraph's own Turtle writer: flat statements. serialize(..., pretty=True) groups and nests (see the Graphs guide).
print(g.serialize(format="turtle"))

triples: 3
@prefix ex: <http://example.org/> .
ex:alice ex:knows ex:bob ;
	ex:name "Alice" ;
	a ex:Person .



## 3. Querying a `StarLayerGraph`

The `.query()` function sends the SPARQL text to Oxigraph and returns a `Result` that iterates rows. Row values are pyoxigraph terms: `str()` of a `NamedNode` is its N-Triples form (`<http://...>`), and `.value` is the bare IRI. See the [SPARQL guide](03-sparql.ipynb) for RDF-1.2-aware query capabilities.

In [4]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

rows = g.query("""
    PREFIX ex: <http://example.org/>
    SELECT ?name WHERE { ?person ex:knows ?name }
""")
for row in rows:
    print(row.name, "|", row.name.value, "|", g.qname_term(row.name))

<http://example.org/bob> | http://example.org/bob | ex:bob


## 4. Validation with SHACL

A SHACL shape graph can be used to describe the constraints within a graph. StarLayer validates with its own SHACL engine, `starlayer.shacl.validate(data_graph, shacl_graph=...)`, which accepts `StarLayerGraph` and `StarLayerDataset` directly.

See the [SHACL shapes guide](04-shacl-shapes.ipynb) for `starlayer.shacl`'s additions to reflect SHACL 1.2 and how to use the `StarShaclSchema` object.

In [5]:
data = StarLayerGraph()
data.bind("ex", EX)
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

result = validate(data, shacl_graph=shapes)
print("conforms:", result.conforms)
print(result.report_text)

conforms: True
Validation Report
Conforms: True

